<h1 style="color:#d53a49;font-weight:600;">Sentiment Analysis with Hotel Reviews</h1>

> **OBJECTIVE:** Clean the data a bit more; add columns that will be useful later, change the values in other columns, drop certain columns completely.
>

---

<h3 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">1 | Initial Column Processing</h3>

- Drop `lat` and `long`; we will not use them.
- Replace `Hotel_Address` values with the following shortened more useful values:
    - Amsterdam, Netherlands
    - Barcelona, Spain
    - London, United Kingdom
    - Milan, Italy
    - Paris, France
    - Vienna, Austria

In [63]:
import pandas as pd
import time
import ast

start = time.time()
df = pd.read_csv("../data/Hotel_Reviews.csv")


def replace_address(row):
    if "Netherlands" in row["Hotel_Address"]:
        return "Amsterdam, Netherlands"
    elif "Barcelona" in row["Hotel_Address"]:
        return "Barcelona, Spain"
    elif "United Kingdom" in row["Hotel_Address"]:
        return "London, United Kingdom"
    elif "Milan" in row["Hotel_Address"]:
        return "Milan, Italy"
    elif "France" in row["Hotel_Address"]:
        return "Paris, France"
    elif "Vienna" in row["Hotel_Address"]:
        return "Vienna, Austria"
    else:
        return row["Hotel_Address"]


df.drop(["lat", "lng"], axis=1, inplace=True)

df["Hotel_Address"] = df.apply(replace_address, axis=1)

The sum of the value_counts(), should add up to the total number of reviews:

In [64]:
print(df["Hotel_Address"].value_counts().sum(), "reviews.")

515738 reviews.


Now, we can query country level data:

In [65]:
display(df.groupby("Hotel_Address").agg({"Hotel_Name": "nunique"}))

,Hotel_Name
Hotel_Address,
"Amsterdam, Netherlands",105
"Barcelona, Spain",211
"London, United Kingdom",400
"Milan, Italy",162
"Paris, France",458
"Vienna, Austria",158


<h3 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">2 | Process Hotel Meta_Review Columns</h3>

- Drop `Additional_Number_of_Scoring`
- Replace `Total_Number_of_Reviews` with the actual number of reviews in the dataset
- Replace `Average_Score` with our own calculated score.

In [66]:
df.drop(["Additional_Number_of_Scoring"], axis=1, inplace=True)
df.Total_Number_of_Reviews = df.groupby("Hotel_Name")["Hotel_Name"].transform("count")
df.Average_Score = round(df.groupby("Hotel_Name")["Reviewer_Score"].transform("mean"), 1)

<h3 style="font-weight:600;color:white;background:#d53a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">3 | Process Review Columns</h3>

- *Drop* the following columns:
    - `Review_Total_Negative_Word_Counts`
    - `Review_Total_Positive_Word_Counts`
    - `Review_Date`
    - `days_since_review`
- Keep the following columns:
    - `Reviewer_Score`
    - `Negative_Review`
    - `Positive_Review`
- Keep `Tags` for now until further processing
- *Drop* `Total_Number_of_Reviews_Reviewer_Has_Given`
- Keep `Reviewer_Nationality`

In [67]:
cols_to_drop = [
    "Review_Total_Negative_Word_Counts",
    "Review_Total_Positive_Word_Counts",
    "Review_Date",
    "days_since_review",
    "Total_Number_of_Reviews_Reviewer_Has_Given",
]
df.drop(cols_to_drop, axis=1, inplace=True)

<h3 style="font-weight:600;color:white;background:#d57a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">4 | Tag Columns</h3>

To ascertain the tags of interest, we must follow a few steps:
- The **type of trip** should stay
- The **type of guest group** should stay
- The type of room is irrelevant
- The device the review was submitted, is irrelevant
- The number of nights stayed is probably irrelevant

To count the tags in a better format, first remove opening and closing brackets and then remove all quotes too.

In [68]:
df["Tags"] = df["Tags"].str.strip("[']")
df["Tags"] = df["Tags"].str.replace(" ', '", ",", regex=False)

Finally we create new columns for each of the most relevant tags. For every review row, if the `Tag` column matches one of the new columns, add a 1, if not, add a 0. The end result will be a count of how many reviewers chose this hotel for e.g. business vs leisure, or to bring a pet.

In [69]:
df["Leisure_trip"] = df.Tags.apply(lambda tag: 1 if "Leisure trip" in tag else 0)
df["Couple"] = df.Tags.apply(lambda tag: 1 if "Couple" in tag else 0)
df["Solo_traveler"] = df.Tags.apply(lambda tag: 1 if "Solo traveler" in tag else 0)
df["Business_trip"] = df.Tags.apply(lambda tag: 1 if "Business trip" in tag else 0)
df["Group"] = df.Tags.apply(lambda tag: 1 if "Group" in tag or "Travelers with friends" in tag else 0)
df["Family_with_young_children"] = df.Tags.apply(lambda tag: 1 if "Family with young children" in tag else 0)
df["Family_with_older_children"] = df.Tags.apply(lambda tag: 1 if "Family with older children" in tag else 0)
df["With_a_pet"] = df.Tags.apply(lambda tag: 1 if "With a pet" in tag else 0)

<h3 style="font-weight:600;color:white;background:#d57a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">5 | Save your file</h3>

In [70]:
print("Saving results to Hotel_Reviews_Filtered.csv")
df.to_csv(r"../data/Hotel_Reviews_Filtered.csv", index=False)

Saving results to Hotel_Reviews_Filtered.csv


<h3 style="font-weight:600;color:white;background:#d57a49;padding:25px;margin:20;text-align:left;display:fill;border-radius:10px 0 0 10px;text-transform:uppercase;">6 | Sentiment Analysis Operations</h3>

In [71]:
import time
import pandas as pd
import nltk as nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download("vader_lexicon")
nltk.download("stopwords")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\nkmil\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nkmil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Load the filtered hotel reviews from CSV

In [72]:
df = pd.read_csv("../data/Hotel_Reviews_Filtered.csv")

<span style="color:#d57a49;text-transform:uppercase;font-weight:600;">1. Removing Stop Words</span>

Stop words are common English words that do not change the sentiment of a sentence. Removing them, makes the sentiment analysis run faster while retaining its accuracy!

So, remove them from both columns:

In [73]:
start = time.time()
cache = set(stopwords.words("english"))


def remove_stopwords(review):
    text = " ".join([word for word in review.split() if word not in cache])
    return text


df.Negative_Review = df.Negative_Review.apply(remove_stopwords)
df.Positive_Review = df.Positive_Review.apply(remove_stopwords)

print(f"Removal of stop words took: {(time.time()-start):.2f} secs.")

Removal of stop words took: 2.40 secs.


<span style="color:#d57a49;text-transform:uppercase;font-weight:600;">2. Performing Sentiment Analysis</span>

Calculate sentiment analysis for both columns, and store the results in two new columns. Then we can compare these values with the actual reviewer's score.

In [74]:
vader_sentiment = SentimentIntensityAnalyzer()


def calc_sentiment(review):
    if review == "No Negative" or review == "No Positive":
        return 0
    return vader_sentiment.polarity_scores(review)["compound"]


print("Calculating sentiment columns for both positive and negative reviews")
start = time.time()
df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)
end = time.time()
print(f"Calculating sentiment took: {round(end-start,2)} secs.")

Calculating sentiment columns for both positive and negative reviews
Calculating sentiment took: 102.6 secs.


To test if the sentiment matches the review:

In [75]:
df = df.sort_values(by=["Negative_Sentiment"], ascending=True)
print(df[["Negative_Review", "Negative_Sentiment"]])

df = df.sort_values(by=["Positive_Sentiment"], ascending=True)
print(df[["Positive_Review", "Positive_Sentiment"]])

                                          Negative_Review  Negative_Sentiment
186584  So bad experience memories I hotel The first n...             -0.9920
129503  First charged twice room booked booking second...             -0.9896
307286  The staff Had bad experience even booking Janu...             -0.9889
201953  Everything DO NOT STAY AT THIS HOTEL I never i...             -0.9886
452092  No WLAN room Incredibly rude restaurant staff ...             -0.9884
...                                                   ...                 ...
138365  Wifi terribly slow I speed test network upload...              0.9938
79215   I find anything hotel first I walked past hote...              0.9938
278506  The property great location There bakery next ...              0.9945
339189  Guys I like hotel I wish return next year Howe...              0.9948
480509  I travel lot far visited countless number hote...              0.9957

[515738 rows x 2 columns]
                                     

<span style="color:#d57a49;text-transform:uppercase;font-weight:600;">3. Save the file</span>

In [78]:
df = df.reindex(
    [
        "Hotel_Name",
        "Hotel_Address",
        "Total_Number_of_Reviews",
        "Average_Score",
        "Reviewer_Score",
        "Negative_Sentiment",
        "Positive_Sentiment",
        "Reviewer_Nationality",
        "Leisure_trip",
        "Couple",
        "Solo_traveler",
        "Business_trip",
        "Group",
        "Family_with_young_children",
        "Family_with_older_children",
        "With_a_pet",
        "Negative_Review",
        "Positive_Review",
    ],
    axis=1,
)

print("Saving results to Hotel_Reviews_NLP.csv")
df.to_csv(r"../data/Hotel_Reviews_NLP.csv", index=False)

Saving results to Hotel_Reviews_NLP.csv
